# Notebook 04 - Feature Validation

Notebook này thực hiện quá trình đánh giá và lựa chọn các đặc trưng có mức độ liên quan cao đến Digital Burnout nhằm xác định bộ biến đầu vào phù hợp cho quá trình xây dựng mô hình học máy.

Quy trình Feature Validation được thực hiện thông qua hai nhóm phương pháp:

- Statistical Feature Assessment: Sử dụng ANOVA F-test để đánh giá khả năng phân biệt của từng biến đối với biến mục tiêu.
- Machine Learning Feature Assessment: Sử dụng Random Forest Feature Importance để đánh giá mức độ đóng góp của từng đặc trưng trong mô hình học máy.

Kết quả từ hai phương pháp được so sánh nhằm xác định các đặc trưng quan trọng và xây dựng tập biến cuối cùng phục vụ Modeling.

# 0. Setup

Thiết lập môi trường thực thi cho Notebook 04 - Feature Validation, bao gồm khai báo thư viện cần thiết, cấu hình hiển thị dữ liệu và chuẩn bị các công cụ phục vụ quá trình đánh giá đặc trưng.

In [1]:
# Import các thư viện xử lý dữ liệu

import pandas as pd
import numpy as np

# Import các thư viện trực quan hóa dữ liệu

import matplotlib.pyplot as plt
import seaborn as sns

# Import các phương pháp thống kê

from scipy.stats import f_oneway

# Import các công cụ xử lý dữ liệu

from sklearn.preprocessing import LabelEncoder

# Import mô hình học máy phục vụ đánh giá đặc trưng

from sklearn.ensemble import RandomForestClassifier

# Import công cụ hỗ trợ hiển thị kết quả

from IPython.display import display

# Thiết lập cấu hình hiển thị dữ liệu

pd.set_option(
    "display.max_columns",
    None
)
pd.set_option(
    "display.max_rows",
    100
)

# Thiết lập kích thước mặc định cho biểu đồ

plt.rcParams["figure.figsize"] = (10, 5)

print("Đã thiết lập môi trường phân tích.")

Đã thiết lập môi trường phân tích.


# 1. Load Cleaned Dataset

Tải bộ dữ liệu khảo sát đã được tiền xử lý từ Notebook 02 nhằm sử dụng làm đầu vào cho quá trình Feature Validation.

In [5]:
# Khai báo đường dẫn tương đối đến dataset đã xử lý

data_path = "../../data/processed/vietnam_dataset/vn_digital_burnout_cleaned.csv"

# Đọc dataset đã được xử lý

survey_dataset = pd.read_csv(
    data_path,
    encoding="utf-8-sig"
)

# Kiểm tra thông tin dataset

print("Đã tải dữ liệu thành công.")
print(f"Số lượng quan sát: {survey_dataset.shape[0]:,}")
print(f"Số lượng biến: {survey_dataset.shape[1]:,}")

Đã tải dữ liệu thành công.
Số lượng quan sát: 654
Số lượng biến: 30


# 2. Feature Validation Configuration

Xây dựng cấu hình đánh giá đặc trưng dựa trên framework Digital Burnout Index (DBI), nhằm xác định các nhóm biến đầu vào và biến mục tiêu phục vụ quá trình Feature Validation.

Phần này thực hiện:

- Xác định các nhóm chỉ báo trong framework DBI.
- Xác định biến mục tiêu cho bài toán đánh giá.
- Phân loại biến đầu vào theo kiểu dữ liệu:
    - Numerical Features.
    - Categorical Features.

## 2.1 Research Feature Groups

Xác định các nhóm biến đầu vào dựa trên framework Digital Burnout Index đã được xây dựng trong nghiên cứu.

Các biến được phân nhóm thành:

- Demographic Variables: Các đặc điểm cá nhân của người tham gia.
- Context Variables: Bối cảnh học tập/làm việc và sử dụng thiết bị.
- Digital Exposure Indicators: Các chỉ báo phản ánh mức độ tiếp xúc với môi trường số.
- Cognitive Performance Indicators: Các chỉ báo phản ánh khả năng tập trung và duy trì hiệu suất.
- Sleep & Recovery Indicators: Các chỉ báo phản ánh khả năng phục hồi.
- Digital Burnout Scale Items: Các biến triệu chứng burnout.

Các Composite Indicators và biến mục tiêu không được xem là feature đầu vào trong bước validation.

In [6]:
# Khai báo nhóm biến theo framework Digital Burnout Index

feature_groups = {

    "Demographic Variables": [
        "gender",
        "birth_year",
        "education_stage"
    ],

    "Context Variables": [
        "work_mode",
        "device_usage_type"
    ],

    "Digital Exposure Indicators": [
        "daily_screen_time",
        "social_media_hours",
        "doomscrolling_duration",
        "late_night_device_usage",
        "notification_count",
        "smartphone_unlocks",
        "app_switch_frequency"
    ],

    "Cognitive Performance Indicators": [
        "concentration_score",
        "distraction_frequency",
        "focus_sessions",
        "deep_work_hours",
        "task_completion_rate",
        "motivation_level"
    ],

    "Sleep & Recovery Indicators": [
        "sleep_hours",
        "sleep_quality"
    ],

    "Digital Burnout Scale Items": [
        "digital_exhaustion",
        "digital_stress",
        "physical_fatigue",
        "emotional_exhaustion",
        "performance_decline",
        "loss_of_interest"
    ]
}


# Kiểm tra số lượng biến trong từng nhóm

feature_group_summary = pd.DataFrame({
    "Feature Group": feature_groups.keys(),
    "Number of Variables": [
        len(features)
        for features in feature_groups.values()
    ]
})


display(
    feature_group_summary
)

,Feature Group,Number of Variables
0,Demographic Variables,3
1,Context Variables,2
2,Digital Exposure Indicators,7
3,Cognitive Performance Indicators,6
4,Sleep & Recovery Indicators,2
5,Digital Burnout Scale Items,6


## 2.2 Target Variables

Xác định các biến mục tiêu phục vụ quá trình đánh giá mức độ liên quan của các đặc trưng đối với Digital Burnout.

Trong nghiên cứu này, biến mục tiêu được xây dựng từ framework Digital Burnout Index:

- digital_burnout_score:
    - Chỉ số tổng hợp đại diện cho mức độ Digital Burnout.
    - Được sử dụng làm biến mục tiêu chính trong quá trình đánh giá đặc trưng.

- dbi_level:
    - Nhãn phân loại mức độ Digital Burnout.
    - Bao gồm ba nhóm:
        - Low.
        - Moderate.
        - High.

Trong Notebook 04, digital_burnout_score được sử dụng để xác định mức độ liên quan của các biến đầu vào, trong khi dbi_level được sử dụng để kiểm tra cấu trúc phân loại của dữ liệu.

In [7]:
# Khai báo các biến mục tiêu sử dụng trong Feature Validation

target_variables = [
    "digital_burnout_score",
    "dbi_level"
]

# Kiểm tra thông tin các biến mục tiêu

target_summary = pd.DataFrame({
    "Target Variable": target_variables,
    "Data Type": [
        survey_dataset[column].dtype
        for column in target_variables
    ],
    "Missing Values": [
        survey_dataset[column].isnull().sum()
        for column in target_variables
    ]
})

display(
    target_summary
)

,Target Variable,Data Type,Missing Values
0,digital_burnout_score,float64,0
1,dbi_level,object,0


In [8]:
# Kiểm tra phân bố mức Digital Burnout

dbi_level_distribution = (
    survey_dataset["dbi_level"]
    .value_counts()
    .reset_index()
)

dbi_level_distribution.columns = [
    "DBI Level",
    "Count"
]

display(
    dbi_level_distribution
)

,DBI Level,Count
0,Moderate,445
1,High,147
2,Low,62


## 2.3 Numerical Features

Xác định các biến đầu vào có kiểu dữ liệu số nhằm chuẩn bị cho quá trình đánh giá đặc trưng bằng phương pháp thống kê và học máy.

Các biến Numerical Features bao gồm:

- Các chỉ báo được thu thập dưới dạng thang điểm định lượng.
- Các biến đã được mã hóa số trong quá trình tiền xử lý.

Các biến thuộc nhóm Composite Indicators không được đưa vào danh sách Numerical Features vì đây là các biến tổng hợp được xây dựng từ các chỉ báo đầu vào.

In [9]:
# Xác định các biến đầu vào dạng số

numerical_features = []

# Duyệt qua các nhóm feature đã khai báo

for feature_list in feature_groups.values():
    
    for feature in feature_list:
        
        if pd.api.types.is_numeric_dtype(
            survey_dataset[feature]
        ):
            
            numerical_features.append(feature)

# Tạo bảng tổng hợp numerical features

numerical_summary = pd.DataFrame({
    "Numerical Feature": numerical_features,
    "Data Type": [
        survey_dataset[feature].dtype
        for feature in numerical_features
    ],
    "Unique Values": [
        survey_dataset[feature].nunique()
        for feature in numerical_features
    ]
})

display(
    numerical_summary
)

,Numerical Feature,Data Type,Unique Values
0,daily_screen_time,int64,5
1,social_media_hours,int64,4
2,doomscrolling_duration,int64,5
3,late_night_device_usage,int64,4
4,notification_count,int64,4
5,smartphone_unlocks,int64,4
6,app_switch_frequency,int64,4
7,concentration_score,int64,10
8,distraction_frequency,int64,4
9,focus_sessions,int64,4


## 2.4 Categorical Features

Xác định các biến đầu vào dạng phân loại (categorical features) trong dataset nhằm chuẩn bị cho quá trình đánh giá đặc trưng và lựa chọn phương pháp xử lý phù hợp.

Các biến Categorical Features bao gồm:

- Các biến mô tả đặc điểm nhóm hoặc bối cảnh của người tham gia.
- Các biến có giá trị dạng nhãn thay vì giá trị số liên tục.

Trong framework Digital Burnout Index, nhóm biến này chủ yếu bao gồm:

- Demographic Variables:
    - gender.
    - education_stage.

- Context Variables:
    - work_mode.
    - device_usage_type.

Các biến categorical không được đưa trực tiếp vào ANOVA F-test mà sẽ được xử lý phù hợp trong các bước đánh giá bằng mô hình học máy.

In [10]:
# Xác định các biến đầu vào dạng categorical

categorical_features = []

# Duyệt qua các nhóm feature đã khai báo

for feature_list in feature_groups.values():

    for feature in feature_list:

        if pd.api.types.is_object_dtype(
            survey_dataset[feature]
        ):

            categorical_features.append(feature)

# Tạo bảng tổng hợp categorical features

categorical_summary = pd.DataFrame({
    "Categorical Feature": categorical_features,
    "Data Type": [
        survey_dataset[feature].dtype
        for feature in categorical_features
    ],
    "Unique Values": [
        survey_dataset[feature].nunique()
        for feature in categorical_features
    ]
})

display(
    categorical_summary
)

,Categorical Feature,Data Type,Unique Values
0,gender,object,2
1,birth_year,object,5
2,education_stage,object,7
3,work_mode,object,4
4,device_usage_type,object,4


## 2.5 Validation Configuration Summary

Tổng hợp cấu hình Feature Validation trước khi thực hiện các phương pháp đánh giá đặc trưng, đảm bảo các nhóm biến đầu vào và biến mục tiêu đã được xác định đầy đủ.

Phần này tổng hợp:

- Các nhóm feature trong framework Digital Burnout Index.
- Số lượng biến thuộc từng nhóm.
- Danh sách biến Numerical Features.
- Danh sách biến Categorical Features.
- Biến mục tiêu sử dụng trong quá trình đánh giá.

Việc tổng hợp cấu hình giúp kiểm tra tính nhất quán của dữ liệu trước khi thực hiện ANOVA F-test và Machine Learning Feature Assessment.

In [11]:
# Tổng hợp cấu hình Feature Validation

validation_summary = pd.DataFrame({
    "Configuration": [
        "Feature Groups",
        "Numerical Features",
        "Categorical Features",
        "Target Variables"
    ],
    "Count": [
        len(feature_groups),
        len(numerical_features),
        len(categorical_features),
        len(target_variables)
    ]
})

display(
    validation_summary
)

,Configuration,Count
0,Feature Groups,6
1,Numerical Features,21
2,Categorical Features,5
3,Target Variables,2


In [12]:
# Tổng hợp toàn bộ feature đầu vào

validated_input_features = (
    numerical_features +
    categorical_features
)


print(f"Số lượng biến đầu vào: {len(validated_input_features)}")


# Kiểm tra các biến bị trùng lặp

duplicate_features = (
    pd.Series(validated_input_features)
    .value_counts()
)

duplicate_features = duplicate_features[
    duplicate_features > 1
]


display(
    duplicate_features
)

Số lượng biến đầu vào: 26


Series([], Name: count, dtype: int64)

# 3. ANOVA F-test

Đánh giá mức độ khác biệt của các Numerical Features giữa các nhóm Digital Burnout Level, từ đó xác định các biến có khả năng liên quan đến sự khác biệt về mức độ Digital Burnout.

Phương pháp ANOVA F-test được sử dụng để kiểm tra giả thuyết:

- H0: Giá trị trung bình của feature giữa các nhóm DBI Level không có sự khác biệt đáng kể.
- H1: Có ít nhất một nhóm DBI Level có giá trị trung bình khác biệt.

Trong nghiên cứu này:

- Biến phân nhóm: dbi_level
- Biến đánh giá: Numerical Features

Các biến có giá trị F-statistic cao và p-value thấp cho thấy có khả năng phân biệt tốt hơn giữa các nhóm Digital Burnout.

## 3.1 Perform ANOVA Test

Thực hiện kiểm định ANOVA F-test để đánh giá sự khác biệt về giá trị trung bình của từng Numerical Feature giữa các nhóm mức độ Digital Burnout.

ANOVA F-test được áp dụng nhằm xác định liệu các biến đầu vào có khả năng phân biệt giữa các nhóm Digital Burnout Level hay không.

Quy trình thực hiện:

- Phân chia dữ liệu theo từng nhóm:
    - Low.
    - Moderate.
    - High.
- Tính toán giá trị F-statistic cho từng Numerical Feature.
- Xác định mức ý nghĩa thống kê thông qua p-value.

Diễn giải kết quả:

- F-statistic càng lớn: Feature có sự khác biệt lớn hơn giữa các nhóm DBI Level.

- p-value < 0.05: Feature có sự khác biệt có ý nghĩa thống kê.

- p-value ≥ 0.05: Chưa có đủ bằng chứng cho thấy feature khác biệt giữa các nhóm.

In [13]:
# Khai báo biến nhóm phục vụ ANOVA

anova_target = "dbi_level"

# Khởi tạo danh sách lưu kết quả ANOVA

anova_results = []

# Thực hiện ANOVA cho từng numerical feature

for feature in numerical_features:

    group_values = [
        survey_dataset[
            survey_dataset[anova_target] == group
        ][feature]
        for group in survey_dataset[anova_target].unique()
    ]

    f_statistic, p_value = f_oneway(
        *group_values
    )

    anova_results.append({
        "Feature": feature,
        "F-Statistic": f_statistic,
        "P-Value": p_value
    })

# Chuyển kết quả ANOVA sang DataFrame

anova_results = pd.DataFrame(
    anova_results
)

# Sắp xếp feature theo F-statistic giảm dần

anova_results = anova_results.sort_values(
    by="F-Statistic",
    ascending=False
)

display(
    anova_results
)

,Feature,F-Statistic,P-Value
15,digital_exhaustion,77.528278,6.289798e-31
16,digital_stress,74.983400,4.944099e-30
19,performance_decline,70.502365,1.926286e-28
17,physical_fatigue,61.058877,4.973406e-25
18,emotional_exhaustion,45.823784,2.403380e-19
3,late_night_device_usage,33.264722,1.757036e-14
20,loss_of_interest,31.369879,9.848678e-14
0,daily_screen_time,16.873216,7.172476e-08
6,app_switch_frequency,11.260380,1.557277e-05
2,doomscrolling_duration,9.181246,1.169054e-04


## 3.2 Select Significant Features

Lựa chọn các đặc trưng có ý nghĩa thống kê dựa trên kết quả ANOVA F-test để xác định các biến có khả năng phân biệt giữa các nhóm Digital Burnout Level.

Các feature được xem là có ý nghĩa thống kê khi thỏa điều kiện:

- P-value < 0.05.

Các feature này cho thấy giá trị trung bình có sự khác biệt đáng kể giữa các nhóm:

- Low.
- Moderate.
- High.

Kết quả ANOVA được sử dụng như một bước đánh giá ban đầu. Các feature được lựa chọn sẽ tiếp tục được so sánh với kết quả Feature Importance từ mô hình học máy nhằm xây dựng bộ đặc trưng cuối cùng.

In [14]:
# Lọc các feature có ý nghĩa thống kê dựa trên p-value

significant_features = (
    anova_results[
        anova_results["P-Value"] < 0.05
    ]
    ["Feature"]
    .tolist()
)

# Hiển thị số lượng feature có ý nghĩa thống kê

print(
    f"Số lượng feature có ý nghĩa thống kê: {len(significant_features)}"
)

# Hiển thị danh sách feature

significant_feature_table = pd.DataFrame({
    "Significant Feature": significant_features
})

display(
    significant_feature_table
)

Số lượng feature có ý nghĩa thống kê: 14


,Significant Feature
0,digital_exhaustion
1,digital_stress
2,performance_decline
3,physical_fatigue
4,emotional_exhaustion
5,late_night_device_usage
6,loss_of_interest
7,daily_screen_time
8,app_switch_frequency
9,doomscrolling_duration


# 4. Machine Learning Feature Assessment

Đánh giá mức độ quan trọng của các đặc trưng bằng phương pháp học máy nhằm bổ sung cho kết quả Feature Validation từ ANOVA F-test.

Trong phần này, mô hình Random Forest được sử dụng như một phương pháp đánh giá Feature Importance.

Khác với ANOVA F-test chỉ xem xét mối quan hệ đơn biến giữa từng feature và biến mục tiêu, Random Forest có khả năng:

- Xem xét đồng thời nhiều đặc trưng trong quá trình phân loại.
- Phát hiện các mối quan hệ phi tuyến giữa feature và Digital Burnout Level.
- Đánh giá mức độ đóng góp của từng biến trong quá trình dự đoán.

## 4.1 Prepare Dataset for Machine Learning Feature Assessment

Chuẩn bị dữ liệu đầu vào cho mô hình Random Forest nhằm đánh giá mức độ quan trọng của các đặc trưng.

Trong bước này, dataset được chuyển đổi thành hai thành phần:

- Feature Matrix (X): Bao gồm các Numerical Features thuộc framework Digital Burnout Index.
- Target Variable (y): Biến phân loại `dbi_level`.

Biến mục tiêu được giữ ở dạng ba lớp:

- Low.
- Moderate.
- High.

In [15]:
# Khai báo feature matrix và target variable

ml_features = numerical_features.copy()

target_column = "dbi_level"

# Tạo dữ liệu đầu vào cho mô hình học máy

X = survey_dataset[
    ml_features
]

y = survey_dataset[
    target_column
]

# Kiểm tra thông tin dữ liệu đầu vào

print(f"Số lượng biến đầu vào: {X.shape[1]}")
print(f"Số lượng quan sát: {X.shape[0]:,}")

# Kiểm tra phân bố biến mục tiêu

target_distribution = (
    y.value_counts()
    .reset_index()
)

target_distribution.columns = [
    "DBI Level",
    "Count"
]

display(
    target_distribution
)

Số lượng biến đầu vào: 21
Số lượng quan sát: 654


,DBI Level,Count
0,Moderate,445
1,High,147
2,Low,62


## 4.2 Train Random Forest Model

Huấn luyện mô hình Random Forest để đánh giá mức độ đóng góp của từng đặc trưng trong việc phân loại mức độ Digital Burnout.

Random Forest được lựa chọn vì có khả năng:

- Xử lý tốt dữ liệu dạng bảng.
- Đánh giá mức độ quan trọng của từng biến đầu vào.
- Phát hiện các mối quan hệ phi tuyến giữa feature và target.

Trong bước này:

- Feature đầu vào: Numerical Features.
- Biến mục tiêu: dbi_level.
- Mô hình chỉ được sử dụng để trích xuất Feature Importance, không dùng để đánh giá hiệu suất dự đoán.

Các tham số mô hình được thiết lập nhằm đảm bảo tính tái lập kết quả:

- random_state = 42.

In [16]:
# Khởi tạo mô hình Random Forest phục vụ đánh giá Feature Importance

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

# Huấn luyện mô hình trên tập dữ liệu

random_forest_model.fit(
    X,
    y
)

print("Đã hoàn thành huấn luyện mô hình Random Forest.")

Đã hoàn thành huấn luyện mô hình Random Forest.


In [17]:
# Kiểm tra số lượng cây trong mô hình

print(
    f"Số lượng cây trong mô hình: {len(random_forest_model.estimators_)}"
)

Số lượng cây trong mô hình: 200


## 4.3 Extract Random Forest Feature Importance

Trích xuất và xếp hạng mức độ quan trọng của các đặc trưng dựa trên kết quả từ mô hình Random Forest.

Random Forest đánh giá mức độ quan trọng của feature dựa trên mức độ đóng góp của từng biến trong quá trình xây dựng các cây quyết định.

Feature Importance càng cao cho thấy biến đó có vai trò lớn hơn trong việc phân loại mức độ Digital Burnout.

Quy trình thực hiện:

- Lấy giá trị feature importance từ mô hình Random Forest.
- Ghép với tên các feature tương ứng.
- Sắp xếp theo mức độ quan trọng giảm dần.
- Trực quan hóa các feature có mức độ ảnh hưởng cao nhất.

In [18]:
# Trích xuất mức độ quan trọng của từng feature

feature_importance_results = pd.DataFrame({
    "Feature": ml_features,
    "Importance": random_forest_model.feature_importances_
})

# Sắp xếp feature theo mức độ quan trọng giảm dần

feature_importance_results = (
    feature_importance_results
    .sort_values(
        by="Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

# Hiển thị bảng Feature Importance

display(
    feature_importance_results
)

,Feature,Importance
0,loss_of_interest,0.108078
1,digital_exhaustion,0.099671
2,digital_stress,0.085296
3,physical_fatigue,0.082970
4,performance_decline,0.079153
5,emotional_exhaustion,0.074050
6,motivation_level,0.046895
7,concentration_score,0.046215
8,sleep_quality,0.045819
9,sleep_hours,0.034686


# 5. Compare ANOVA and Machine Learning Results

So sánh kết quả đánh giá đặc trưng từ hai phương pháp khác nhau nhằm xác định các feature có mức độ quan trọng ổn định đối với Digital Burnout.

Trong nghiên cứu này, hai phương pháp được sử dụng với mục đích bổ sung cho nhau:

- ANOVA F-test:
    - Đánh giá mức độ khác biệt thống kê của từng feature giữa các nhóm Digital Burnout Level.
    - Dựa trên F-statistic và p-value.

- Random Forest Feature Importance:
    - Đánh giá mức độ đóng góp của feature trong quá trình phân loại.
    - Dựa trên khả năng dự đoán của mô hình học máy.

Việc kết hợp hai phương pháp giúp hạn chế việc lựa chọn feature chỉ dựa trên một góc nhìn duy nhất.

Các feature được xem là có mức độ tin cậy cao khi:

- Có ý nghĩa thống kê trong ANOVA.
- Có mức độ quan trọng trong Random Forest.

In [20]:
# Chuẩn hóa tên cột trước khi kết hợp kết quả

anova_comparison = anova_results[
    [
        "Feature",
        "F-Statistic",
        "P-Value"
    ]
]

rf_comparison = feature_importance_results[
    [
        "Feature",
        "Importance"
    ]
]

# Kết hợp kết quả từ hai phương pháp

feature_comparison = pd.merge(
    anova_comparison,
    rf_comparison,
    on="Feature",
    how="inner"
)

# Xếp hạng theo hai phương pháp

feature_comparison["ANOVA_Rank"] = (
    feature_comparison["F-Statistic"]
    .rank(
        ascending=False,
        method="dense"
    )
)


feature_comparison["RF_Rank"] = (
    feature_comparison["Importance"]
    .rank(
        ascending=False,
        method="dense"
    )
)

# Tính điểm xếp hạng trung bình

feature_comparison["Average_Rank"] = (
    feature_comparison["ANOVA_Rank"] +
    feature_comparison["RF_Rank"]
) / 2

# Sắp xếp feature theo Average Rank

feature_comparison = (
    feature_comparison
    .sort_values(
        by="Average_Rank"
    )
    .reset_index(drop=True)
)

display(
    feature_comparison
)

,Feature,F-Statistic,P-Value,Importance,ANOVA_Rank,RF_Rank,Average_Rank
0,digital_exhaustion,77.528278,6.289798e-31,0.099671,1.0,2.0,1.5
1,digital_stress,74.983400,4.944099e-30,0.085296,2.0,3.0,2.5
2,performance_decline,70.502365,1.926286e-28,0.079153,3.0,5.0,4.0
3,physical_fatigue,61.058877,4.973406e-25,0.082970,4.0,4.0,4.0
4,loss_of_interest,31.369879,9.848678e-14,0.108078,7.0,1.0,4.0
5,emotional_exhaustion,45.823784,2.403380e-19,0.074050,5.0,6.0,5.5
6,motivation_level,5.478569,4.370078e-03,0.046895,11.0,7.0,9.0
7,doomscrolling_duration,9.181246,1.169054e-04,0.031570,10.0,11.0,10.5
8,sleep_quality,4.092018,1.713709e-02,0.045819,12.0,9.0,10.5
9,daily_screen_time,16.873216,7.172476e-08,0.028639,8.0,16.0,12.0


In [21]:
# Xác định feature xuất hiện trong nhóm có ý nghĩa thống kê và nhóm quan trọng từ Random Forest

top_rf_features = (
    feature_importance_results
    .head(14)["Feature"]
    .tolist()
)


validated_overlap_features = list(
    set(significant_features)
    .intersection(
        top_rf_features
    )
)


# Hiển thị danh sách feature giao nhau

print(
    f"Số lượng feature được cả hai phương pháp lựa chọn: {len(validated_overlap_features)}"
)


display(
    pd.DataFrame({
        "Validated Feature": validated_overlap_features
    })
)

Số lượng feature được cả hai phương pháp lựa chọn: 10


,Validated Feature
0,motivation_level
1,loss_of_interest
2,digital_exhaustion
3,emotional_exhaustion
4,notification_count
5,performance_decline
6,physical_fatigue
7,sleep_quality
8,digital_stress
9,doomscrolling_duration


# 6. Save Validated Features

Lưu lại danh sách các đặc trưng đã được xác thực sau quá trình Feature Validation để sử dụng cho các bước xây dựng mô hình dự đoán Digital Burnout trong Notebook tiếp theo.

Sau khi kết hợp kết quả từ:

- ANOVA F-test.
- Random Forest Feature Importance.
- Cơ sở lý thuyết của Digital Burnout Index Framework.

Một tập hợp feature cuối cùng được lựa chọn nhằm đảm bảo:

- Có ý nghĩa thống kê.
- Có mức độ đóng góp trong mô hình học máy.
- Có khả năng giải thích về mặt nghiên cứu.

Danh sách feature này sẽ được sử dụng làm đầu vào cho quá trình Modeling thay vì sử dụng toàn bộ biến ban đầu.

In [22]:
# Khai báo danh sách feature đã được xác thực

validated_features_list = [
    "motivation_level",
    "loss_of_interest",
    "digital_exhaustion",
    "emotional_exhaustion",
    "notification_count",
    "performance_decline",
    "physical_fatigue",
    "sleep_quality",
    "digital_stress",
    "doomscrolling_duration"
]

# Tạo dataframe lưu validated features

validated_features = pd.DataFrame({
    "Feature": validated_features_list
})


# Khai báo đường dẫn lưu file

output_path = "../../data/processed/vietnam_dataset/validated_features.csv"

# Lưu danh sách feature

validated_features.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)


print("Đã lưu danh sách feature được xác thực.")

Đã lưu danh sách feature được xác thực.
